# Tutorial: Training a Convolutional Neural Network (CNN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/adversarial2dEnvAI/blob/master/notebooks/CNN_Training.ipynb)

This notebook guides you through the process of training a simple CNN to classify images of dogs and flowers. These images are used in the `CustomGrid` environment to demonstrate how an agent can use computer vision to perceive its surroundings.

## 1. Introduction to CNNs

A **Convolutional Neural Network (CNN)** is a type of deep learning model specifically designed for processing structured arrays of data, such as images. 

### Key Layers:
- **Convolutional Layer (Conv2D)**: Applies filters to the image to extract features like edges, corners, and textures.
- **Pooling Layer (MaxPooling2D)**: Reduces the spatial dimensions (width and height) of the data, making the model more efficient and robust to small translations.
- **Flatten Layer**: Converts the 2D feature maps into a 1D vector to be passed into fully connected layers.
- **Dense Layer**: Fully connected layers that perform the final classification based on the extracted features.

### Metrics:
- **Loss**: Measures how far the model's predictions are from the actual labels. We want to minimize this.
- **Accuracy**: The percentage of correctly classified images.

## 2. Setup

First, we need to install the `custom_grid_env` package and its dependencies.

In [ ]:
!pip install git+https://github.com/dgaida/adversarial2dEnvAI.git

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from custom_grid_env.cnn_tutorial.data_generation import generate_data

## 3. Data Generation

We will generate a synthetic dataset of dogs and flowers with various backgrounds (white, red crosshatch, green crosshatch) to make our model robust.

In [ ]:
DATA_DIR = "data"
generate_data(output_dir=DATA_DIR, num_samples_per_class=500)

def load_data(data_dir="data"):
    images = []
    labels = []
    class_names = ["dog", "flower"]
    
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        for img_name in os.listdir(class_dir):
            if img_name.endswith(".png"):
                img_path = os.path.join(class_dir, img_name)
                img = keras.utils.load_img(img_path, target_size=(64, 64))
                img_array = keras.utils.img_to_array(img)
                images.append(img_array)
                labels.append(label)
                
    return np.array(images) / 255.0, np.array(labels), class_names

X, y, class_names = load_data(DATA_DIR)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Loaded {len(X)} images.")

### Visualize the Data
Let's look at some examples from our dataset.

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.show()

## 4. Model Definition

We define a simple CNN architecture using Keras.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(64, 64, 3)),
    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

model.summary()

## 5. Training

We train the model for 10 epochs.

In [ ]:
history = model.fit(
    X_train, y_train, 
    epochs=10, 
    validation_data=(X_val, y_val)
)

## 6. Evaluation

Let's visualize the training progress and the final performance.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()

In [ ]:
y_pred = np.argmax(model.predict(X_val), axis=1)
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

## 7. Saving the Model

Finally, we save the model to a file.

In [ ]:
model.save("model.keras")
print("Model saved as model.keras")